In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
thetas = np.linspace(-np.pi, np.pi, 8, endpoint=False) + np.pi / 8
r = 1
xs = r * np.cos(thetas)
ys = r * np.sin(thetas)
Ps = np.column_stack((xs, ys))
capacitance = np.abs(thetas)


def center_of_pressure(pos, mag):
    return pos.T @ mag / np.sum(mag)


fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot(xs, ys, "ok", label="sensor locations")
ax.plot(xs, ys, capacitance, "o", label="measured values")

for x, y, cap in zip(xs, ys, capacitance):
    ax.plot([x, x], [y, y], [0, cap], "k--", alpha=0.5)
cop = center_of_pressure(Ps, capacitance)
ax.plot(cop[0], cop[1], "x", label="center of pressure")
ax.plot(0, 0, "xk", label="centroid")
ax.legend()

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("displacement")

In [ ]:
thetas = np.linspace(-np.pi, np.pi, 8, endpoint=False) + np.pi / 8
r = 1
xs = r * np.cos(thetas)
ys = r * np.sin(thetas)
Ps = np.column_stack((xs, ys))
capacitance = np.abs(thetas)


def best_plane(pos, mag):
    data = np.column_stack((pos, mag))
    X_centered = data - np.mean(data, axis=0)
    _, _, Vt = np.linalg.svd(X_centered)
    return Vt[-1]


def plot_plane(points, normal, ax=None, size=1.0):
    """
    points: (N, 3) array - used to center the plane
    normal: (3,) unit normal vector
    """
    if ax is None:
        fig = plt.figure()
        ax = fig.add_subplot(111, projection="3d")

    centroid = points.mean(axis=0)
    cx, cy, cz = centroid
    nx, ny, nz = normal

    # Build two orthogonal vectors in the plane
    u = np.array([1, 0, 0]) if abs(nx) < 0.9 else np.array([0, 1, 0])
    u = np.cross(normal, u)
    u /= np.linalg.norm(u)
    v = np.cross(normal, u)

    # Mesh over the plane
    s, t = np.meshgrid(np.linspace(-size, size, 10), np.linspace(-size, size, 10))
    plane = centroid + s[..., None] * u + t[..., None] * v

    ax.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], alpha=0.4)
    # ax.scatter(*points.T, color='red', s=20)

    return ax


fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.plot(xs, ys, "ok", label="sensor locations")
ax.plot(xs, ys, capacitance, "o", label="measured values")
ax.plot(0, 0, "xk", label="centroid")

for x, y, cap in zip(xs, ys, capacitance):
    ax.plot([x, x], [y, y], [0, cap], "k--", alpha=0.5)

plane = best_plane(Ps, capacitance)
print(plane)
vec = np.vstack((plane, np.zeros(3)))
ax.plot(*-vec.T, label="plane normal")
plot_plane(np.column_stack((Ps, capacitance)), plane, ax, 2)

cross_prod = np.cross(plane, [0, 0, 1])
rot_vec = np.vstack((cross_prod, np.zeros(3)))
ax.plot(*rot_vec.T, "c", label="rotation vector")

ax.legend(loc="upper right")


ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("displacement")
ax.set_aspect("equal")